# GCP Vertex AI - Triton Inference Server on a Custom Container Endpoint

This notebook deploys ONNX models to **NVIDIA Triton Inference Server** running on a GCP Vertex AI Endpoint.

## Architecture

Vertex AI custom-container deployments consist of three resources:

| Resource | Purpose |
|---|---|
| **Model** | References the Triton Docker image (Artifact Registry) and the GCS model repository. Container args (`--strict-model-config=false`, `--vertex-ai-default-model`) are set here. |
| **Endpoint** | An empty placeholder URL - no compute attached (similar to Azure). |
| **Deployment** | Binds Model to Endpoint on a GPU machine (`n1-standard-4` + `NVIDIA_TESLA_T4`). This provisions the actual compute. |

Adding a new model **requires a redeployment** - Triton loads the model repository from GCS at startup.

Inference uses `endpoint.raw_predict` with the `X-Vertex-Ai-Triton-Redirect` header to route to a specific Triton model. Pixel data is rounded to 3 decimals to stay under Vertex AI's 1.5 MB limit.

## Prerequisites

- GCP project with Vertex AI API enabled.
- Triton image pushed to Artifact Registry:
  ```bash
  gcloud artifacts repositories create triton-repo --repository-format=docker --location=<REGION>
  docker pull nvcr.io/nvidia/tritonserver:24.09-py3
  docker tag  nvcr.io/nvidia/tritonserver:24.09-py3 <REGION>-docker.pkg.dev/<PROJECT>/triton-repo/tritonserver:24.09-py3
  docker push <REGION>-docker.pkg.dev/<PROJECT>/triton-repo/tritonserver:24.09-py3

- Models uploaded to GCS:
  ```bash
  gcloud storage cp -r ./models3/* gs://<BUCKET>/triton/models

- Python packages: google-cloud-aiplatform, numpy, pillow, transformers.

## 1. Imports

In [ ]:
import json
import random
import numpy as np
from google.cloud import aiplatform
from PIL import Image
from transformers import AutoConfig, AutoImageProcessor

## 2. Configuration


In [ ]:
PROJECT_ID = "PROJECT_ID"
REGION = "REGION"
BUCKET_NAME = "BUCKET_NAME"

MODEL_NAME = f"model-{random.randint(0, 10000)}"
ENDPOINT_NAME = f"endpoint-{random.randint(0, 10000)}"

TRITON_IMAGE_TAG = "24.09-py3"
MACHINE_TYPE = "n1-standard-4"
ACCELERATOR_TYPE = "NVIDIA_TESLA_T4"
ACCELERATOR_COUNT = 1
TEST_IMAGE_PATH = "./test_image.jpg"

MODELS = {
    "resnet50_onnx": {
        "hf_model": "microsoft/resnet-50",
        "input_name": "pixel_values",
        "output_name": "logits",
    },
    "convnext_tiny_onnx": {
        "hf_model": "facebook/convnext-tiny-224",
        "input_name": "pixel_values",
        "output_name": "logits",
    },
    "mobilenet_v2_onnx": {
        "hf_model": "google/mobilenet_v2_1.0_224",
        "input_name": "pixel_values",
        "output_name": "logits",
        "processor_kwargs": {"crop_size": {"height": 256, "width": 256}},
    },
}

## 3. Helpers

In [ ]:
def softmax(logits: np.ndarray) -> np.ndarray:
    exp = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    return exp / np.sum(exp, axis=-1, keepdims=True)


def decode_top1(logits: np.ndarray, hf_model: str) -> tuple[str, float]:
    config = AutoConfig.from_pretrained(hf_model)
    probs = softmax(logits)
    pred_idx = int(np.argmax(probs, axis=-1)[0])
    score = float(probs[0, pred_idx])
    label = config.id2label.get(pred_idx, f"class_{pred_idx}")
    return label, score


def triton_image_uri() -> str:
    return f"{REGION}-docker.pkg.dev/{PROJECT_ID}/triton-repo/tritonserver:{TRITON_IMAGE_TAG}"

## 4. Initialize Vertex AI

In [ ]:
aiplatform.init(project=PROJECT_ID, location=REGION)

## 5. Upload Model to Vertex AI

The model references the Triton image in Artifact Registry, the GCS model repository, and container args for Triton.

In [ ]:
model = aiplatform.Model.upload(
    display_name=MODEL_NAME,
    artifact_uri=f"gs://{BUCKET_NAME}/triton/models",
    serving_container_image_uri=triton_image_uri(),
    serving_container_ports=[8000],
    serving_container_predict_route="/v2/models",
    serving_container_health_route="/v2/health/ready",
    serving_container_args=[
        "--strict-model-config=false",
        "--vertex-ai-default-model=resnet50_onnx",
    ],
)
model.wait()

## 6. Create Endpoint & Deploy

- **Endpoint** - empty placeholder (no GPU yet, similar to Azure)
- **Deploy** - provisions a GPU machine that merges model + endpoint

In [ ]:
# Create an empty endpoint (placeholder - no compute)
endpoint = aiplatform.Endpoint.create(display_name=ENDPOINT_NAME)

In [ ]:
# Deploy the model to the endpoint on a GPU machine
endpoint.deploy(
    model=model,
    machine_type=MACHINE_TYPE,
    accelerator_type=ACCELERATOR_TYPE,
    accelerator_count=ACCELERATOR_COUNT,
    min_replica_count=1,
    max_replica_count=1,
    traffic_percentage=100,
)

## 7. Inference

We use `endpoint.raw_predict` with `X-Vertex-Ai-Triton-Redirect` to route each request to the target Triton model.

In [ ]:
img = Image.open(TEST_IMAGE_PATH).convert("RGB")

for model_name_triton, cfg in MODELS.items():
    # Client-side preprocessing with HuggingFace
    processor_kwargs = cfg.get("processor_kwargs", {})
    processor = AutoImageProcessor.from_pretrained(cfg["hf_model"], **processor_kwargs)
    inputs = processor(images=img, return_tensors="np")
    pixel_values = inputs["pixel_values"].astype(np.float32)

    # Build Triton V2 inference request
    # Round to 3 decimals to keep JSON payload under Vertex AI's 1.5 MB limit
    data_list = [round(float(x), 3) for x in pixel_values.flatten()]
    payload = {
        "inputs": [
            {
                "name": cfg["input_name"],
                "shape": list(pixel_values.shape),
                "datatype": "FP32",
                "data": data_list,
            }
        ],
        "outputs": [{"name": cfg["output_name"]}],
    }

    # X-Vertex-Ai-Triton-Redirect selects the target model within Triton
    response = endpoint.raw_predict(
        body=json.dumps(payload, separators=(",", ":")).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "X-Vertex-Ai-Triton-Redirect": f"v2/models/{model_name_triton}/infer",
        },
    )

    result = json.loads(response.text)
    logits = np.array(result["outputs"][0]["data"]).reshape(
        result["outputs"][0]["shape"]
    )

    label, score = decode_top1(logits, cfg["hf_model"])
    print(f"{model_name_triton:<30s}  label={label:<40s}  score={score:.4f}")

## 8. Clean up resources

Run this section when you no longer need the endpoint.

In [ ]:
delete = False

In [ ]:
if delete:
    endpoint.undeploy_all()
    endpoint.delete()
    model.delete()